# 2Baseline Model: Historical, Parametric, and Quantile Benchmarks

This notebook produces strictly out-of-sample benchmark forecasts for the next-month stock return distribution. It is a **model-production notebook**, not an asset-pricing notebook: no portfolio return is used to choose a model here.

The notebook contains two benchmark families:

1. historical empirical VaR/ES;
2. rolling Gaussian;
3. rolling Student-$t$ using rolling moments;
4. conditional one-component Normal and Student-$t$ neural networks using the same predictors as the MDN;
5. joint linear quantile regression and an optional quantile MLP.

All models return the same economic units: raw decimal monthly returns, where `0.05` means 5%. To match `3ES_Calculation.ipynb`, `var_5` is the raw 5% return quantile while `es_5` is the positive loss magnitude $-E[R\mid R\leq VaR_{5\%}]$.


| 类别 | 模型 | 作用 |
|---|---|---|
| 简单历史基准 | Rolling Normal | 检查只使用个股历史均值和波动率能做到什么程度 |
| 线性非参数基准 | Linear Quantile | 检查线性条件分位数模型是否足够 |
| 非线性非参数基准 | Quantile MLP | 检查不假定分布的神经网络是否优于 MDN |
| 单分布神经网络 | Fused Normal K1 | 使用全部特征，但没有 mixture |
| 普通 MDN | Fused Normal K3 | 检查 mixture 本身的增量价值 |
| 结构化 MDN | Structured Normal K3 | 检查宏观—微观结构限制的增量价值 |

## 1. Assumed Input Contract

The cleaned input is assumed to contain one unique row per `(permno, mthcaldt)` and the following fields:

- `permno`: stock identifier;
- `mthcaldt`: forecast formation month $t$;
- `mthret`: raw return observed in month $t$, available when the forecast is formed;
- `target_ret_final`: raw return in month $t+1$;
- cleaned numeric micro and macro predictors observed by month $t$.

The real run reads `micro_data.parquet` and `MacroFactor_McCracken.parquet` separately, merges them by formation month, and uses the explicit feature contract in the configuration cell. `target_ret_final` is asserted to be label-only.


In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

env = 'local'
# env = 'colab'
# env = 'kaggle'

if env == 'kaggle':
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirname, filename))

    micro_df = pd.read_parquet('/kaggle/input/datasets/jianbinchenuc/mdndata-1990/micro_data.parquet')
    macro_df = pd.read_parquet('/kaggle/input/datasets/jianbinchenuc/mdndata-1990/MacroFactor_McCracken.parquet')
    path = '/kaggle/working'

elif env == 'colab':
    # Colab 版本的代码
    # 1. 挂载 Google Drive
    import google.colab 
    # 1. 设置环境变量
    google.colab.drive.mount('/content/drive')
    os.environ['KAGGLE_USERNAME'] = 'jianbinchenuc'
    os.environ['KAGGLE_KEY'] = 'fa590c922d3624b8e0734dc7a90dd95a'

    # 2. 安装 kaggle 库 (如果尚未安装)
    !pip install -q kaggle

    # 3. 创建目录并下载数据集
    # -p 参数指定下载目录，--unzip 参数会自动解压
    !mkdir -p /content/mdndata
    !kaggle datasets download -d jianbinchenuc/mdndata-1990 -p /content/mdndata --unzip

    # 4. 验证文件是否已存在
    print("已下载的文件：", os.listdir('/content/mdndata'))
    micro_df = pd.read_parquet('/content/mdndata/micro_data.parquet') # 可以根据需要切换到 Colab 的路径
    macro_df = pd.read_parquet('/content/mdndata/MacroFactor_McCracken.parquet') # 可以根据需要切换到 Colab 的路径
    path = '/content/drive/MyDrive/Colab Notebooks/'

elif env == 'local':
    micro_df = pd.read_parquet('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/micro_data.parquet') # local 路径
    macro_df = pd.read_parquet('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/MacroFactor_McCracken.parquet') # local 路径
    path = '/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output'

else:
    raise ValueError("Invalid environment specified. Please choose 'kaggle', 'colab', or 'local'.")

In [ ]:
display(pd.concat([micro_df.head(3), micro_df.tail(3)]))
display(pd.concat([macro_df.head(3), macro_df.tail(3)]))

,permno,mthcaldt,mthret,mthprc,mthcap,siccd,gvkey,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,de_ratio,cash_ratio,curr_ratio,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,ffi49,target_ret_final,MOM12m,mu_bench,sigma_bench,mthcap_log
0,10001,1991-01-31,0.013158,9.625,1.014475e+04,4920.0,012994,0.896385,5.690808,3.307560,0.445982,4.872598,14.070388,0.044401,0.099749,0.138304,0.082956,0.112235,0.112708,0.161306,0.392076,2.430626,0.061202,0.672101,1.796365,0.316439,1.340247,1.166318,8.789413,10.931698,3.261452,0.000000,0.0,0.0,-0.055775,31,0.012987,0.009425,0.002110,0.022682,9.224712
1,10001,1991-02-28,0.012987,9.750,1.027650e+04,4920.0,012994,0.975332,5.299653,3.037383,0.434947,5.205927,14.253121,0.047361,0.103356,0.141152,0.085157,0.121406,0.116860,0.168351,0.378677,2.249325,0.050190,0.614188,1.755852,0.289359,1.348320,1.192696,8.491285,10.618904,3.286892,0.000000,0.0,0.0,-0.042869,31,-0.012244,0.029179,0.003717,0.022716,9.237615
2,10001,1991-03-31,-0.012244,9.500,1.001300e+04,4920.0,012994,0.975332,5.299653,2.959502,0.423795,5.072442,13.887656,0.047361,0.103356,0.141152,0.085157,0.121406,0.116860,0.168351,0.378677,2.249325,0.050190,0.614188,1.755852,0.289359,1.348320,1.192696,8.491285,10.618904,3.286892,0.000000,0.0,0.0,-0.042869,31,0.039474,0.029348,0.001628,0.022954,9.211640
1560357,93436,2024-10-31,-0.045025,249.850,8.020335e+08,3711.0,184996,0.105166,57.713252,69.210526,8.406271,69.482219,369.690039,0.131885,0.075809,0.229558,0.183071,0.217032,0.200551,0.207100,0.095088,0.822338,0.293684,0.409772,0.696792,1.017029,1.760274,0.902169,26.475384,5.220111,1.543949,0.047001,0.0,0.0,0.010123,23,0.381469,0.302679,0.027542,0.140071,20.502661
1560358,93436,2024-11-30,0.381469,345.160,1.107984e+09,3711.0,184996,0.083346,53.762150,93.286486,11.394010,76.450592,306.628288,0.132692,0.084756,0.234822,0.183922,0.208738,0.190668,0.203448,0.099842,0.839369,0.274887,0.497103,0.690047,1.047903,1.798235,0.866390,25.589359,5.285830,1.475261,0.044858,0.0,0.0,-0.014637,23,0.170008,0.040696,0.043050,0.167882,20.825808
1560359,93436,2024-12-31,0.170008,403.840,1.298958e+09,3711.0,184996,0.083346,53.762150,109.145946,13.368497,89.698836,274.076290,0.132692,0.084756,0.234822,0.183922,0.208738,0.190668,0.203448,0.099842,0.839369,0.274887,0.497103,0.690047,1.047903,1.798235,0.866390,25.589359,5.285830,1.475261,0.044858,0.0,0.0,-0.014637,23,0.001882,0.389088,0.054301,0.171772,20.984828


,mthcaldt,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD
0,1991-01-31,-0.001000,0.1,0.023836,-0.005920,-0.000778,0.040338,0.02,6.876265,0.001402,-0.069067,0.000930,-0.50,-0.11,0.1,0.001485,22.5845,3.12,1.34,2.35
1,1991-02-28,-0.001892,0.1,-0.041536,-0.004931,-0.003955,-0.029746,0.02,6.682109,0.001999,-0.093898,-0.007471,-0.40,-0.70,-0.3,-0.000014,26.9318,3.54,1.87,2.36
2,1991-03-31,-0.002384,0.2,0.091651,-0.007933,-0.009491,0.005330,0.02,6.872128,0.000704,0.100378,0.005295,-0.66,-0.61,0.5,-0.002975,22.1126,3.82,1.91,2.22
405,2024-10-31,0.001598,-0.1,-0.029656,-0.002832,0.003543,-0.007867,0.01,7.210818,-0.001050,-0.031354,0.002598,-0.20,-0.26,1.6,-0.000073,17.6597,0.29,-1.00,1.70
406,2024-11-30,0.000075,0.0,0.055299,-0.002629,0.001225,0.002425,0.00,7.178545,0.000319,0.023649,-0.000157,-0.30,-0.24,0.3,0.000641,19.9478,0.80,-0.41,1.53
407,2024-12-31,0.001426,0.1,-0.078988,-0.001472,0.002754,-0.010575,0.00,7.161622,0.002139,0.024533,-0.002535,-0.19,-0.09,0.8,0.000686,15.9822,1.13,-0.06,1.42


In [2]:
from pathlib import Path
import importlib.util
import copy
import json
import math
import os
import random
import sys
import warnings

import numpy as np
import pandas as pd
from scipy.stats import norm, t as student_t

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

# ==========================================
# 全局随机种子锁死函数
# ==========================================
def seed_everything(seed=42):
    """
    全局锁死随机种子，确保深度学习模型的绝对可复现性
    """
    # 1. 锁死 Python 内置随机库
    random.seed(seed)
    
    # 2. 锁死 Python 哈希表的随机化 (保证字典遍历顺序一致)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 3. 锁死 Numpy 随机种子
    np.random.seed(seed)
    
    # 4. 锁死 PyTorch CPU 随机种子
    torch.manual_seed(seed)
    
    # 5. 锁死 PyTorch GPU 随机种子
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # 如果你有双卡/多卡
        
        # 【极其关键】锁死 CuDNN 底层算法的随机性
        # 注意：设置为 True 可能会略微牺牲一点点 GPU 训练速度，但为了发论文复现，这是必须的！
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    print(f"Global seed set to {seed} to ensure reproducibility.")

# ==========================================
# 0. 自动设备检测
# ==========================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")


Using device: mps


In [5]:
macro_df.columns

Index(['mthcaldt', 'PAYEMS', 'UNRATE', 'CLAIMSx', 'INDPRO', 'DPCERA3M086SBEA', 'AMDMNOx', 'ISRATIOx', 'HOUST', 'M2SL', 'NONBORRES', 'BUSLOANS', 'FEDFUNDS', 'CP3Mx', 'UEMPMEAN',
       'CPIAUCSL', 'VIXCLSx', 'BAAFFM', 'TERM_SPREAD', 'CREDIT_SPREAD'],
      dtype='object')

In [ ]:
# ALPHAS = (0.01, 0.05, 0.10)
# ALPHA_SUFFIX = {0.01: "1", 0.05: "5", 0.10: "10"}


# # ---------- Time design ----------
# INITIAL_TRAIN_MONTHS = 180
# REFIT_MONTHS = 12
# VALIDATION_MONTHS = 12
# MIN_NEURAL_TRAIN_MONTHS = 36
# QUICK_MAX_STOCKS = 250
# QUICK_OOS_MONTHS = 24

# # ---------- Historical baselines ----------
# HISTORY_WINDOW_MONTHS = INITIAL_TRAIN_MONTHS + VALIDATION_MONTHS
# MIN_HISTORY_MONTHS = 0
# T_DF_MIN = 4.1
# T_DF_MAX = 100.0

# # ---------- Neural baselines ----------
# STANDARDIZE_FEATURES_WITH_TRAINING_DATA = True
# SIGMA_FLOOR = 0.005  # raw-return units: 0.5% monthly volatility
# NU_MIN = 3.0
# NU_MAX = 30.0
# HIDDEN_DIMS = (64, 32)
# DROPOUT = 0.10
# BATCH_SIZE = 4096
# LEARNING_RATE = 1e-3
# WEIGHT_DECAY = 1e-5
# EARLY_STOPPING_PATIENCE = 8
# SEED = 42

# RUN_HISTORICAL_BASELINES = True
# RUN_CONDITIONAL_NORMAL_K1 = True
# RUN_CONDITIONAL_STUDENT_K1 = True
# RUN_LINEAR_QUANTILE = True
# RUN_QUANTILE_MLP = False

# ==========================================
# Rolling Normal Baseline
# ==========================================

In [ ]:
# ==========================================
# Rolling Normal Baseline
# ==========================================
# 对每只股票，使用形成月 t 及以前的历史月收益率，
# 估计 Normal distribution 的均值和标准差，
# 并预测 t+1 月收益率的分布。
#
# 注意：
# 1. mthret 是形成月 t 已经实现的收益率，可以进入历史窗口。
# 2. target_ret_final 是 t+1 月收益率，只作为真实 label，
#    不参与均值和标准差估计。
# 3. ES 保存为正的 loss-ES，和 MDN_ES_Calculation 的定义一致。
# ==========================================


# 防止某只股票的历史标准差为0
ROLLING_SCALE_FLOOR = 1e-6

def normal_var_and_loss_es(alpha, loc, scale):
    """
    计算 Normal distribution 的左尾 VaR 和 loss-ES。

    参数
    ----------
    alpha:
        左尾概率，例如0.01、0.05、0.10。
    loc:
        Normal distribution 的均值。
    scale:
        Normal distribution 的标准差。

    返回
    ----------
    var_value:
        return-space VaR，通常为负数。
    es_loss:
        正的损失幅度。
        如果 return-space ES = -0.20，则 es_loss = 0.20。
    """
    scale = max(float(scale), ROLLING_SCALE_FLOOR)

    # 标准正态分布的 alpha 分位数
    z_alpha = norm.ppf(alpha)

    # VaR_alpha = mu + sigma * Phi^{-1}(alpha)
    var_value = loc + scale * z_alpha

    # E[Y | Y <= VaR_alpha]
    es_return = loc - scale * norm.pdf(z_alpha) / alpha

    # 保存为正的损失幅度
    es_loss = -es_return

    return float(var_value), float(es_loss)



In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

def run_rolling_normal(
    panel,
    max_history_months=INITIAL_TRAIN_MONTHS + VALIDATION_MONTHS,
):
    """
    对每只股票计算 Rolling Normal 样本外预测 (极简重构版)
    """
    # 1. 确保数据按股票和时间排序，这对 rolling 计算至关重要
    panel = panel.sort_values(["permno", "mthcaldt"]).copy()
    
    # 2. 提前计算好【每个月全市场的横截面标准差】
    # 以备有些股票当月只有 1 条记录算不出标准差时作为备用（Fallback）
    cross_sectional_std = panel.groupby("mthcaldt")["mthret"].std(ddof=1)
    
    # 3. 使用 pandas 强大的 rolling 函数直接计算均值和标准差
    # window=最大允许月数, min_periods=1 保证哪怕只有 1 个月也能出结果
    grouped_ret = panel.groupby("permno")["mthret"]
    
    panel["pred_mean"] = grouped_ret.rolling(window=max_history_months, min_periods=1).mean().reset_index(level=0, drop=True)
    panel["pred_vol"] = grouped_ret.rolling(window=max_history_months, min_periods=1).std(ddof=1).reset_index(level=0, drop=True)
    panel["history_n"] = grouped_ret.rolling(window=max_history_months, min_periods=1).count().reset_index(level=0, drop=True)
    
    # 4. 重点：处理历史数据只有 1 个月导致标准差为 NaN 的情况
    # 做法：将该月的全市场标准差映射过去填补空缺
    panel["pred_vol"] = panel["pred_vol"].fillna(panel["mthcaldt"].map(cross_sectional_std))
    
    # 结合波动率下限限制保护（防止极其罕见的波动率为0导致数学崩溃）
    panel["pred_vol"] = panel["pred_vol"].clip(lower=ROLLING_SCALE_FLOOR)
    
    # 5. 圈定样本外（OOS）数据
    available_dates = pd.DatetimeIndex(np.sort(panel["mthcaldt"].unique()))
    OOS_START = available_dates[max_history_months]
    oos_panel = panel[panel["mthcaldt"] >= OOS_START].copy()

    # 6. 生成最终结果字典 (移除了繁琐的二分查找，直接迭代计算好的 dataframe)
    output_rows = []
    
    for _, row in tqdm(oos_panel.iterrows(), total=len(oos_panel), desc="Rolling Normal"):
        forecast_date = pd.Timestamp(row["mthcaldt"])
        
        output_row = {
            "permno": row["permno"],
            "forecast_date": forecast_date,
            "target_date": forecast_date + pd.offsets.MonthEnd(1),
            "realized_ret": float(row["target_ret_final"]),
            "model_id": "rolling_normal",
            
            # 统计分布参数
            "pred_mean": float(row["pred_mean"]),
            "pred_vol": float(row["pred_vol"]),
            "loc": float(row["pred_mean"]),
            "scale": float(row["pred_vol"]),
            "history_n": int(row["history_n"]),
        }

        # 计算不同 alpha 下的 VaR 和 ES
        for alpha in ALPHAS:
            suffix = ALPHA_SUFFIX[alpha]
            var_value, es_loss = normal_var_and_loss_es(
                alpha=alpha,
                loc=output_row["loc"],
                scale=output_row["scale"],
            )
            output_row[f"var_{suffix}"] = var_value
            output_row[f"es_{suffix}"] = es_loss

        output_rows.append(output_row)

    # 组装返回
    rolling_df = pd.DataFrame(output_rows)
    rolling_df = (
        rolling_df
        .sort_values(["forecast_date", "permno"])
        .reset_index(drop=True)
    )

    return rolling_df

# ==========================================
# 生成 Rolling Normal 样本外预测
# ==========================================

rolling_normal_forecasts = run_rolling_normal(
    panel=micro_df,
    max_history_months=INITIAL_TRAIN_MONTHS + VALIDATION_MONTHS
)

display(rolling_normal_forecasts.head())

In [20]:
rolling_normal_summary = (
    rolling_normal_forecasts
    .groupby("forecast_date")
    .agg(
        n_firms=("permno", "size"),
        mean_history_n=("history_n", "mean"),
        mean_predicted_vol=("pred_vol", "mean"),
        mean_es_5=("es_5", "mean"),
    )
)

display(rolling_normal_summary.head())

,n_firms,mean_history_n,mean_predicted_vol,mean_es_5
forecast_date,,,,
2007-01-31,4041,112.406830,0.129376,0.242627
2007-02-28,4040,112.686881,0.128531,0.242056
2007-03-31,4016,112.848108,0.128381,0.241951
2007-04-30,4025,112.664596,0.127501,0.240183
2007-05-31,4026,112.429955,0.127349,0.239534


# ==========================================
# Linear Quantile：线性分位数回归
# ==========================================

In [ ]:
import torch.optim as optim

In [ ]:
ALPHAS = (0.01, 0.05, 0.10)
ALPHA_SUFFIX = {0.01: "1", 0.05: "5", 0.10: "10"}

QUANTILE_LEVELS = np.array([
    0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.10, 0.25, 0.50,
    0.75, 0.90, 0.95, 0.975, 0.99, 0.995, 0.9975, 0.999
], dtype=float)

# # ---------- Time design ----------
INITIAL_TRAIN_MONTHS = 180
TEST_MONTHS = 12
VALIDATION_MONTHS = 12
# MIN_NEURAL_TRAIN_MONTHS = 36
# QUICK_MAX_STOCKS = 250
# QUICK_OOS_MONTHS = 24

# # ---------- Historical baselines ----------
# HISTORY_WINDOW_MONTHS = INITIAL_TRAIN_MONTHS + VALIDATION_MONTHS
# MIN_HISTORY_MONTHS = 0
# T_DF_MIN = 4.1
# T_DF_MAX = 100.0

# # ---------- Neural baselines ----------
# STANDARDIZE_FEATURES_WITH_TRAINING_DATA = True
# SIGMA_FLOOR = 0.005  # raw-return units: 0.5% monthly volatility
# NU_MIN = 3.0
# NU_MAX = 30.0
# HIDDEN_DIMS = (64, 32)
# DROPOUT = 0.10
# BATCH_SIZE = 4096
# LEARNING_RATE = 1e-3
# WEIGHT_DECAY = 1e-5
# EARLY_STOPPING_PATIENCE = 8
# SEED = 42

# RUN_HISTORICAL_BASELINES = True
# RUN_CONDITIONAL_NORMAL_K1 = True
# RUN_CONDITIONAL_STUDENT_K1 = True
# RUN_LINEAR_QUANTILE = True
# RUN_QUANTILE_MLP = False

In [ ]:
# ==========================================
# Linear Quantile：线性分位数回归
# ==========================================
class LinearQuantile(nn.Module):
    def __init__(self, input_dim, quantile_levels):
        super().__init__()
        self.linear = nn.Linear(input_dim, len(quantile_levels))
        self.register_buffer("quantile_levels", torch.tensor(quantile_levels, dtype=torch.float32).view(1, -1))

    def forward(self, x):
        return self.linear(x)


def weighted_pinball_loss(pred, target, weight, quantile_levels):
    error = target - pred
    loss = torch.maximum(quantile_levels * error, (quantile_levels - 1) * error)
    return (loss.mean(dim=1) * weight).sum() / weight.sum().clamp_min(1e-12)

# ==========================================
# 生成与 MDN 相同的滚动窗口
# ==========================================
def rolling_window_generator(panel):
    dates = np.sort(panel["mthcaldt"].unique())

    current_split_idx = INITIAL_TRAIN_MONTHS

    while current_split_idx + VALIDATION_MONTHS < len(dates):
        start_idx = current_split_idx - INITIAL_TRAIN_MONTHS

        train_dates = dates[start_idx:current_split_idx]
        val_dates = dates[current_split_idx:current_split_idx + VALIDATION_MONTHS]
        test_dates = dates[current_split_idx + VALIDATION_MONTHS:current_split_idx + VALIDATION_MONTHS + REFIT_MONTHS]

        train_df = panel[panel["mthcaldt"].isin(train_dates)].copy()
        val_df = panel[panel["mthcaldt"].isin(val_dates)].copy()
        test_df = panel[panel["mthcaldt"].isin(test_dates)].copy()

        yield train_df, val_df, test_df

        current_split_idx += TEST_MONTHS

# ==========================================
# 训练单个 Linear Quantile
# ==========================================
def train_linear_quantile(train_df, val_df):
    feature_mean, feature_std = fit_feature_scaler(train_df)

    X_train = transform_features(train_df, feature_mean, feature_std)
    X_val = transform_features(val_df, feature_mean, feature_std)

    y_train = train_df["target_ret_final"].to_numpy(dtype=np.float32)
    y_val = val_df["target_ret_final"].to_numpy(dtype=np.float32)

    w_train = add_month_equal_weights(train_df).astype(np.float32)
    w_val = add_month_equal_weights(val_df).astype(np.float32)

    train_dataset = TensorDataset(
        torch.tensor(X_train), torch.tensor(y_train), torch.tensor(w_train)
    )
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

    X_val = torch.tensor(X_val, device=device)
    y_val = torch.tensor(y_val, device=device).unsqueeze(1)
    w_val = torch.tensor(w_val, device=device)
    quantiles = torch.tensor(QUANTILE_LEVELS, dtype=torch.float32, device=device).view(1, -1)

    model = LinearQuantile(len(FEATURE_COLS), QUANTILE_LEVELS).to(device)

    # 用训练样本的无条件分位数初始化截距
    initial_quantiles = np.quantile(y_train, QUANTILE_LEVELS)
    with torch.no_grad():
        model.linear.weight.zero_()
        model.linear.bias.copy_(torch.tensor(initial_quantiles, dtype=torch.float32, device=device))

    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    best_val_loss = float("inf")
    best_state = None
    epochs_no_improve = 0

    for epoch in range(MAX_EPOCHS):
        model.train()

        for batch_x, batch_y, batch_w in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device).unsqueeze(1)
            batch_w = batch_w.to(device)

            optimizer.zero_grad()
            pred = model(batch_x)

            pinball = weighted_pinball_loss(pred, batch_y, batch_w, quantiles)
            crossing = torch.relu(pred[:, :-1] - pred[:, 1:]).mean()
            loss = pinball + 0.10 * crossing

            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val)
            val_loss = weighted_pinball_loss(val_pred, y_val, w_val, quantiles).item()

        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
                break

    model.load_state_dict(best_state)
    return model, feature_mean, feature_std, best_val_loss, epoch + 1


# ==========================================
# 根据分位数函数计算均值、波动率、VaR 和 ES
# ==========================================
def add_quantile_distribution_results(output, quantile_pred):
    levels = QUANTILE_LEVELS
    quantile_pred = np.sort(quantile_pred, axis=1)  # 防止分位数交叉
    trapz = getattr(np, "trapezoid", np.trapz)

    full_levels = np.r_[0.0, levels, 1.0]
    full_quantiles = np.column_stack((quantile_pred[:, 0], quantile_pred, quantile_pred[:, -1]))

    pred_mean = trapz(full_quantiles, full_levels, axis=1)
    second_moment = trapz(full_quantiles ** 2, full_levels, axis=1)

    output["pred_mean"] = pred_mean
    output["pred_vol"] = np.sqrt(np.maximum(second_moment - pred_mean ** 2, 0.0))

    for alpha in ALPHAS:
        suffix = ALPHA_SUFFIX[alpha]
        alpha_idx = np.flatnonzero(np.isclose(levels, alpha))[0]
        below_alpha = levels < alpha

        var_value = quantile_pred[:, alpha_idx]
        tail_levels = np.r_[0.0, levels[below_alpha], alpha]
        tail_quantiles = np.column_stack((quantile_pred[:, 0], quantile_pred[:, below_alpha], var_value))

        output[f"var_{suffix}"] = var_value
        output[f"es_{suffix}"] = -trapz(tail_quantiles, tail_levels, axis=1) / alpha

    output["quantile_levels"] = json.dumps(levels.tolist())
    output["quantile_vec"] = [json.dumps(row.tolist()) for row in quantile_pred]

    return output


# ==========================================
# 完整滚动预测
# ==========================================
def run_linear_quantile(panel):
    results = []

    for train_df, val_df, test_df in rolling_window_generator(panel):
        train_start = train_df["mthcaldt"].min().strftime("%Y-%m")
        test_start = test_df["mthcaldt"].min().strftime("%Y-%m")
        print(f"Train starts {train_start} | Test starts {test_start}")

        model, feature_mean, feature_std, val_loss, epochs_run = train_linear_quantile(train_df, val_df)

        test_df = test_df.sort_values(["mthcaldt", "permno"]).copy()
        X_test = transform_features(test_df, feature_mean, feature_std)
        X_test = torch.tensor(X_test, device=device)

        model.eval()
        with torch.no_grad():
            quantile_pred = model(X_test).cpu().numpy()

        output = test_df[["permno", "mthcaldt", "target_ret_final"]].copy()
        output = output.rename(columns={"mthcaldt": "forecast_date", "target_ret_final": "realized_ret"})
        output["target_date"] = output["forecast_date"] + pd.offsets.MonthEnd(1)
        output["model_id"] = "linear_quantile"
        output["validation_loss"] = val_loss
        output["epochs_run"] = epochs_run

        results.append(add_quantile_distribution_results(output, quantile_pred))

    return pd.concat(results, ignore_index=True)


# ==========================================
# 运行
# ==========================================

linear_quantile_forecasts = run_linear_quantile(micro_df)
display(linear_quantile_forecasts.head())

## 2. Configuration

`RUN_MODE='smoke'` uses synthetic data, `quick` uses a bounded stock sample and 24 OOS months, and `full` runs the complete production sample. The environment and paths are detected automatically for local, Colab, and Kaggle runs.


In [ ]:
# ---------- Run mode ----------
RUN_MODE = 'full'  #  'quick', or 'full'

# Explicit feature contract shared with 3MDN_model.ipynb.
MICRO_FEATURES = [
    'mthret', 'mthcap', 'bm', 'evm', 'pe_exi', 'ps', 'pcf', 'pfcf', 'npm', 'opmad', 'gpm', 'cfm', 'roe', 'aftret_invcapx',
       'gprof', 'debt_at', 'debt_ebitda', 'short_debt', 'ocf_lct', 'de_ratio', 'cash_ratio', 'curr_ratio', 'at_turn', 'rect_turn', 'pay_turn', 'sale_equity', 'rd_sale',
       'adv_sale', 'staff_sale', 'accrual', 'MOM12m', 'mu_bench', 'sigma_bench', 'mthcap_log'
]
RAW_SCALE_MICRO_FEATURES = [
    'mthret', 'MOM12m', 'mu_bench', 'sigma_bench', 'mthcap_log',
]
RANK_MICRO_FEATURES = [
    col for col in MICRO_FEATURES if col not in RAW_SCALE_MICRO_FEATURES
]


micro_exclude_cols = {
    'permno', 'mthcaldt', 'mthcap', 'mthprc', 'siccd', 'gvkey', 'ffi49', 'target_ret_final'
}

micro_cols = [
    col for col in micro_df.columns
    if col not in micro_exclude_cols
]

micro_df = micro_df[['permno', 'mthcaldt', 'ffi49', 'target_ret_final'] + micro_cols].copy()
micro_df['mthcaldt'] = pd.to_datetime(micro_df['mthcaldt']) + pd.offsets.MonthEnd(0)


MACRO_FEATURES = [
    'PAYEMS', 'UNRATE', 'CLAIMSx', 'INDPRO', 'DPCERA3M086SBEA', 'AMDMNOx', 'ISRATIOx', 'HOUST', 'M2SL', 'NONBORRES', 'BUSLOANS', 'FEDFUNDS', 'CP3Mx', 'UEMPMEAN',
       'CPIAUCSL', 'VIXCLSx', 'BAAFFM', 'TERM_SPREAD', 'CREDIT_SPREAD',
]

# ---------- Forecast objects ----------
ALPHAS = (0.01, 0.05, 0.10)
ALPHA_SUFFIX = {0.01: '1', 0.05: '5', 0.10: '10'}
QUANTILE_LEVELS = np.array([
    0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.10, 0.25, 0.50,
    0.75, 0.90, 0.95, 0.975, 0.99, 0.995, 0.9975, 0.999
], dtype=float)

# ---------- Time design ----------
INITIAL_TRAIN_MONTHS = 180
REFIT_MONTHS = 12
VALIDATION_MONTHS = 12
MIN_NEURAL_TRAIN_MONTHS = 36
QUICK_MAX_STOCKS = 250
QUICK_OOS_MONTHS = 24

# ---------- Historical baselines ----------
HISTORY_WINDOW_MONTHS = INITIAL_TRAIN_MONTHS + VALIDATION_MONTHS
MIN_HISTORY_MONTHS = 36
T_DF_MIN = 4.1
T_DF_MAX = 100.0

# ---------- Neural baselines ----------
STANDARDIZE_FEATURES_WITH_TRAINING_DATA = True
SIGMA_FLOOR = 0.005  # raw-return units: 0.5% monthly volatility
NU_MIN = 3.0
NU_MAX = 30.0
HIDDEN_DIMS = (64, 32)
DROPOUT = 0.10
BATCH_SIZE = 4096
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
EARLY_STOPPING_PATIENCE = 8
SEED = 42

RUN_HISTORICAL_BASELINES = True
RUN_CONDITIONAL_NORMAL_K1 = True
RUN_CONDITIONAL_STUDENT_K1 = True
RUN_LINEAR_QUANTILE = True
RUN_QUANTILE_MLP = False

if RUN_MODE == 'quick':
    MAX_EPOCHS = 10
else:
    MAX_EPOCHS = 100
    
seed_everything(SEED)

## 3. Load or Generate the Model Panel


In [7]:
micro_df['mthcaldt'] = pd.to_datetime(micro_df['mthcaldt']) + pd.offsets.MonthEnd(0)
macro_df['mthcaldt'] = pd.to_datetime(macro_df['mthcaldt']) + pd.offsets.MonthEnd(0)

# Monthly cross-sectional ranks use only information available in formation month t.
ranked = micro_df.groupby('mthcaldt')[RANK_MICRO_FEATURES].rank(pct=True) * 2.0 - 1.0
micro_df[RANK_MICRO_FEATURES] = ranked.fillna(0.0).to_numpy()
panel_df = micro_df.merge(
    macro_df, on='mthcaldt', how='left', validate='many_to_one'
)

FEATURE_COLS = MICRO_FEATURES + MACRO_FEATURES

panel_df = panel_df.sort_values(['mthcaldt', 'permno']).reset_index(drop=True)
if RUN_MODE == 'quick':
    rng = np.random.default_rng(SEED)
    stock_ids = np.sort(panel_df['permno'].dropna().unique())
    selected_ids = rng.choice(
        stock_ids, size=min(QUICK_MAX_STOCKS, len(stock_ids)), replace=False
    )
    panel_df = panel_df.loc[panel_df['permno'].isin(selected_ids)].copy()

available_dates = pd.DatetimeIndex(sorted(pd.to_datetime(panel_df['mthcaldt']).unique()))

inferred_position = INITIAL_TRAIN_MONTHS + VALIDATION_MONTHS
if inferred_position >= len(available_dates):
    raise ValueError('Not enough months for the requested train/validation design.')
OOS_START = pd.Timestamp(available_dates[inferred_position])
OOS_END = pd.Timestamp(available_dates[-1])
if RUN_MODE == 'quick':
    quick_dates = available_dates[available_dates >= OOS_START][:QUICK_OOS_MONTHS]
    if len(quick_dates):
        OOS_END = min(pd.Timestamp(OOS_END), pd.Timestamp(quick_dates[-1]))
print('Panel shape:', panel_df.shape)
print('Date range:', panel_df['mthcaldt'].min(), 'to', panel_df['mthcaldt'].max())
print('Feature count:', len(FEATURE_COLS))
display(panel_df.head())


Panel shape: (1560360, 60)
Date range: 1991-01-31 00:00:00 to 2024-12-31 00:00:00
Feature count: 53


,permno,mthcaldt,mthret,mthprc,mthcap,siccd,gvkey,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,de_ratio,cash_ratio,curr_ratio,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,ffi49,target_ret_final,MOM12m,mu_bench,sigma_bench,mthcap_log,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD
0,10001,1991-01-31,0.013158,9.625,-0.899905,4920.0,012994,0.218879,-0.536902,-0.756098,-0.483054,-0.351283,0.527399,-0.256889,0.173899,-0.796009,-0.334812,0.159328,0.288565,-0.477352,0.663605,0.424137,-0.615458,0.554007,0.405765,0.273361,-0.590751,0.244853,0.569845,0.356984,0.514729,-0.292683,-0.281596,-0.168514,-0.401330,31,0.012987,0.009425,0.002110,0.022682,9.224712,-0.001,0.1,0.023836,-0.00592,-0.000778,0.040338,0.02,6.876265,0.001402,-0.069067,0.00093,-0.5,-0.11,0.1,0.001485,22.5845,3.12,1.34,2.35
1,10009,1991-01-31,-0.018339,6.500,-0.961989,6020.0,016739,0.005702,-0.009186,0.000950,-0.004118,0.002217,0.032309,0.000317,-0.000950,-0.004118,-0.029458,-0.000634,0.008236,0.001584,0.002217,0.000317,-0.025974,0.035793,0.000317,-0.020272,0.038961,0.002534,0.004118,0.012987,0.002534,-0.292683,-0.281596,-0.168514,-0.002217,0,0.038462,0.096350,0.008446,0.072335,8.718745,-0.001,0.1,0.023836,-0.00592,-0.000778,0.040338,0.02,6.876265,0.001402,-0.069067,0.00093,-0.5,-0.11,0.1,0.001485,22.5845,3.12,1.34,2.35
2,10010,1991-01-31,0.307692,12.750,-0.119417,3840.0,012622,-0.882167,0.863161,0.937282,0.832753,0.958822,0.947418,0.275895,0.175166,0.735192,-0.380424,0.454545,0.683244,0.877732,-0.482420,-0.530567,0.683244,-0.536902,-0.608489,-0.318340,0.574913,0.712385,-0.457080,0.641432,0.228381,0.707317,-0.281596,-0.168514,0.730757,12,0.058824,3.333335,0.205897,0.383231,11.399054,-0.001,0.1,0.023836,-0.00592,-0.000778,0.040338,0.02,6.876265,0.001402,-0.069067,0.00093,-0.5,-0.11,0.1,0.001485,22.5845,3.12,1.34,2.35
3,10016,1991-01-31,-0.036364,13.250,0.185303,3560.0,004641,0.594552,-0.727589,-0.723155,-0.853659,-0.960089,-0.486855,-0.371555,-0.181501,-0.333545,-0.486221,0.375356,0.482420,0.291099,0.669940,0.422236,-0.335445,-0.361419,0.528033,0.321508,-0.173266,0.365220,-0.312639,0.501425,0.683877,0.552106,-0.281596,-0.168514,0.820082,34,0.161039,-0.384369,-0.036227,0.112936,12.101749,-0.001,0.1,0.023836,-0.00592,-0.000778,0.040338,0.02,6.876265,0.001402,-0.069067,0.00093,-0.5,-0.11,0.1,0.001485,22.5845,3.12,1.34,2.35
4,10019,1991-01-31,0.000000,6.500,-0.519797,3610.0,011908,0.272094,-0.408299,-0.658537,-0.187203,-0.337346,0.242952,0.785873,0.632563,0.500792,0.547038,0.741527,-0.789357,0.645233,-0.551473,-0.642065,-0.696547,0.901806,-0.809946,0.814381,0.913209,0.413367,-0.290466,0.915109,-0.427304,0.863161,-0.281596,-0.168514,0.371555,38,0.076923,-0.458333,-0.041975,0.123657,10.479384,-0.001,0.1,0.023836,-0.00592,-0.000778,0.040338,0.02,6.876265,0.001402,-0.069067,0.00093,-0.5,-0.11,0.1,0.001485,22.5845,3.12,1.34,2.35


## 4. Validate Timing and Schema

The checks below are intentionally strict. If the target is not exactly the next observation of the raw stock return, stop and repair the data notebook before training a benchmark.


In [8]:
panel_df = panel_df.copy()
panel_df['mthcaldt'] = pd.to_datetime(panel_df['mthcaldt']) + pd.offsets.MonthEnd(0)
panel_df = panel_df.sort_values(['permno', 'mthcaldt']).reset_index(drop=True)
panel_df['target_date'] = panel_df['mthcaldt'] + pd.offsets.MonthEnd(1)

required_cols = ['permno', 'mthcaldt', 'mthret', 'target_ret_final'] + FEATURE_COLS
missing_cols = [col for col in required_cols if col not in panel_df.columns]
if missing_cols:
    raise KeyError(f'Missing required columns: {missing_cols}')

if panel_df.duplicated(['permno', 'mthcaldt']).any():
    duplicates = panel_df.loc[panel_df.duplicated(['permno', 'mthcaldt'], keep=False), ['permno', 'mthcaldt']]
    raise ValueError(f'Duplicate stock-month rows found. Example: {duplicates.head()}')

expected_target = panel_df.groupby('permno')['mthret'].shift(-1)
next_observed_date = panel_df.groupby('permno')['mthcaldt'].shift(-1)
# The $5 formation-price filter can remove an intermediate row. Validate the
# target against the next row only when that row is the next calendar month.
alignment_mask = (
    panel_df['target_ret_final'].notna()
    & next_observed_date.eq(panel_df['target_date'])
)
max_alignment_error = (
    expected_target.loc[alignment_mask] - panel_df.loc[alignment_mask, 'target_ret_final']
).abs().max()

numeric_block = panel_df[FEATURE_COLS + ['mthret', 'target_ret_final']].replace([np.inf, -np.inf], np.nan)
feature_missing = numeric_block[FEATURE_COLS].isna().sum()
if feature_missing.sum() > 0:
    raise ValueError(f'Features still contain missing values: {feature_missing[feature_missing > 0]}')
return_missing = numeric_block.loc[panel_df['target_ret_final'].notna(), ['mthret', 'target_ret_final']].isna().sum()
if return_missing.sum() > 0:
    raise ValueError(f'Observed current/target returns contain non-finite values: {return_missing}')

panel_df = panel_df.loc[panel_df['target_ret_final'].notna()].copy()
panel_df = panel_df.loc[panel_df['mthcaldt'].between(panel_df['mthcaldt'].min(), OOS_END)].copy()

print('Schema and target alignment checks passed.')
print('OOS period:', OOS_START, 'to', OOS_END)
print('OOS rows:', int((panel_df['mthcaldt'] >= OOS_START).sum()))


Schema and target alignment checks passed.
OOS period: 2007-01-31 00:00:00 to 2024-12-31 00:00:00
OOS rows: 683379


: 

## 5. Distribution and Output Helpers


In [9]:
def safe_scale(value: float, floor: float = 1e-6) -> float:
    return float(max(float(value), floor))

def normal_var_and_loss_es(alpha: float, loc: float, scale: float) -> tuple[float, float]:
    scale = safe_scale(scale)
    z = norm.ppf(alpha)
    var_value = loc + scale * z
    tail_mean = loc - scale * norm.pdf(z) / alpha
    return float(var_value), float(-tail_mean)

def student_var_and_loss_es(
    alpha: float, loc: float, scale: float, degrees_of_freedom: float
) -> tuple[float, float]:
    scale = safe_scale(scale)
    degrees_of_freedom = float(max(degrees_of_freedom, 1.01))
    z = student_t.ppf(alpha, df=degrees_of_freedom)
    density = student_t.pdf(z, df=degrees_of_freedom)
    standard_tail_mean = -(
        (degrees_of_freedom + z ** 2) / (degrees_of_freedom - 1)
    ) * density / alpha
    var_value = loc + scale * z
    tail_mean = loc + scale * standard_tail_mean
    return float(var_value), float(-tail_mean)

def rolling_t_parameters(history: np.ndarray) -> tuple[float, float, float, float]:
    history = np.asarray(history, dtype=float)
    loc = float(np.mean(history))
    sample_std = safe_scale(np.std(history, ddof=1))
    centered = history - loc
    second_moment = float(np.mean(centered ** 2))
    fourth_moment = float(np.mean(centered ** 4))
    excess_kurtosis = fourth_moment / max(second_moment ** 2, 1e-12) - 3.0
    if excess_kurtosis > 0:
        degrees_of_freedom = 6.0 / excess_kurtosis + 4.0
    else:
        degrees_of_freedom = T_DF_MAX
    degrees_of_freedom = float(np.clip(degrees_of_freedom, T_DF_MIN, T_DF_MAX))
    scale = sample_std * math.sqrt((degrees_of_freedom - 2.0) / degrees_of_freedom)
    return loc, safe_scale(scale), degrees_of_freedom, sample_std

def base_forecast_row(row: pd.Series, model_id: str) -> dict:
    return {
        'permno': row['permno'],
        'forecast_date': row['mthcaldt'],
        'target_date': row['target_date'],
        'realized_ret': float(row['target_ret_final']),
        'model_id': model_id,
    }

def add_tail_statistics(output_row: dict, tail_values: dict[float, tuple[float, float]]) -> dict:
    for alpha, (var_value, es_value) in tail_values.items():
        suffix = ALPHA_SUFFIX[alpha]
        output_row[f'var_{suffix}'] = float(var_value)
        output_row[f'es_{suffix}'] = float(es_value)
    return output_row

def empty_forecast_frame() -> pd.DataFrame:
    return pd.DataFrame(columns=[
        'permno', 'forecast_date', 'target_date', 'realized_ret', 'model_id',
        'pred_mean', 'pred_vol', 'var_1', 'es_1',
        'var_5', 'es_5', 'var_10', 'es_10'
    ])


: 

## 6. Historical Empirical, Rolling Normal, and Rolling Student-$t$

All rolling windows end in month $t$ and therefore use only returns known when predicting month $t+1$. The Student-$t$ benchmark uses a scalable method-of-moments estimate of the degrees of freedom instead of fitting a nonlinear MLE for every stock-month.


In [10]:
def empirical_var_and_loss_es(history: np.ndarray, alpha: float) -> tuple[float, float]:
    ordered = np.sort(np.asarray(history, dtype=float))
    var_value = float(np.quantile(ordered, alpha, method='linear'))
    tail_mass = alpha * len(ordered)
    full_observations = int(math.floor(tail_mass))
    fractional_mass = tail_mass - full_observations
    tail_sum = float(ordered[:full_observations].sum())
    if full_observations < len(ordered):
        tail_sum += fractional_mass * float(ordered[full_observations])
    tail_mean = tail_sum / tail_mass
    return var_value, -tail_mean

def run_rolling_distribution_baselines(panel: pd.DataFrame) -> pd.DataFrame:
    output_rows = []

    grouped = panel.groupby('permno', sort=False)
    for _, stock_df in tqdm(grouped, total=panel['permno'].nunique(), desc='Rolling baselines'):
        stock_df = stock_df.sort_values('mthcaldt').reset_index(drop=True)
        stock_returns = stock_df['mthret'].to_numpy(dtype=float)
        stock_dates = pd.to_datetime(stock_df['mthcaldt']).to_numpy()

        for position, row in stock_df.iterrows():
            forecast_date = row['mthcaldt']
            if forecast_date < OOS_START or forecast_date > OOS_END:
                continue

            history_start_date = forecast_date - pd.offsets.MonthEnd(HISTORY_WINDOW_MONTHS - 1)
            history_mask = (stock_dates >= np.datetime64(history_start_date)) & (
                stock_dates <= np.datetime64(forecast_date)
            )
            history = stock_returns[history_mask]
            history = history[np.isfinite(history)]
            if len(history) < MIN_HISTORY_MONTHS:
                continue

            historical_mean = float(np.mean(history))
            historical_std = safe_scale(np.std(history, ddof=1))

            # Historical empirical distribution
            empirical_tail_values = {
                alpha: empirical_var_and_loss_es(history, alpha) for alpha in ALPHAS
            }
            empirical_row = base_forecast_row(row, 'historical_empirical')
            empirical_row.update({
                'pred_mean': historical_mean,
                'pred_vol': historical_std,
                'history_n': int(len(history)),
            })
            output_rows.append(add_tail_statistics(empirical_row, empirical_tail_values))

            # Rolling Gaussian
            normal_tail_values = {
                alpha: normal_var_and_loss_es(alpha, historical_mean, historical_std)
                for alpha in ALPHAS
            }
            normal_row = base_forecast_row(row, 'rolling_normal')
            normal_row.update({
                'pred_mean': historical_mean,
                'pred_vol': historical_std,
                'history_n': int(len(history)),
                'loc': historical_mean,
                'scale': historical_std,
            })
            output_rows.append(add_tail_statistics(normal_row, normal_tail_values))

            # Rolling Student-t by moments
            t_loc, t_scale, t_nu, t_vol = rolling_t_parameters(history)
            student_tail_values = {
                alpha: student_var_and_loss_es(alpha, t_loc, t_scale, t_nu)
                for alpha in ALPHAS
            }
            student_row = base_forecast_row(row, 'rolling_student_t_mom')
            student_row.update({
                'pred_mean': t_loc,
                'pred_vol': t_vol,
                'history_n': int(len(history)),
                'loc': t_loc,
                'scale': t_scale,
                'nu': t_nu,
            })
            output_rows.append(add_tail_statistics(student_row, student_tail_values))

    return pd.DataFrame(output_rows) if output_rows else empty_forecast_frame()

if RUN_HISTORICAL_BASELINES:
    rolling_forecasts = run_rolling_distribution_baselines(panel_df)
else:
    rolling_forecasts = empty_forecast_frame()

print('Rolling forecast rows:', len(rolling_forecasts))
display(rolling_forecasts.groupby('model_id').size().rename('rows').to_frame())


Rolling baselines:   0%|          | 0/16900 [00:00<?, ?it/s]

Rolling forecast rows: 1681773


,rows
model_id,
historical_empirical,560591
rolling_normal,560591
rolling_student_t_mom,560591


: 

## 7. Conditional One-Component Normal and Student-$t$ Networks

These are deliberately simpler same-information-set MLP benchmarks. They use all designated firm and macro predictors but impose a single conditional distribution instead of a mixture. For an exact architecture-matched nested comparison, run `3MDN_model.ipynb` with `best_k=1`. Each refit here uses only rows whose target return was already observable before the first forecast month of the block.


In [ ]:
class ConditionalDistributionNet(nn.Module):
    def __init__(
        self, input_dim: int, distribution: str, hidden_dims=HIDDEN_DIMS, dropout=DROPOUT
    ):
        super().__init__()
        if distribution not in {'normal', 'student_t'}:
            raise ValueError('distribution must be normal or student_t')
        self.distribution = distribution

        layers = []
        previous_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(previous_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.ELU(),
                nn.Dropout(dropout),
            ])
            previous_dim = hidden_dim
        self.backbone = nn.Sequential(*layers)
        self.mu_head = nn.Linear(previous_dim, 1)
        self.sigma_head = nn.Linear(previous_dim, 1)
        self.nu_head = nn.Linear(previous_dim, 1) if distribution == 'student_t' else None

        nn.init.xavier_normal_(self.mu_head.weight, gain=0.1)
        nn.init.zeros_(self.mu_head.bias)
        nn.init.constant_(self.sigma_head.bias, -2.35)
        if self.nu_head is not None:
            nn.init.zeros_(self.nu_head.weight)
            nn.init.zeros_(self.nu_head.bias)

    def forward(self, x):
        hidden = self.backbone(x)
        mu = self.mu_head(hidden).squeeze(-1)
        sigma = F.softplus(self.sigma_head(hidden).squeeze(-1)) + SIGMA_FLOOR
        if self.nu_head is None:
            nu = None
        else:
            raw_nu = self.nu_head(hidden).squeeze(-1)
            nu = NU_MIN + (NU_MAX - NU_MIN) * torch.sigmoid(raw_nu)
        return mu, sigma, nu

def fit_feature_scaler(training_frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    if not STANDARDIZE_FEATURES_WITH_TRAINING_DATA:
        return np.zeros(len(FEATURE_COLS), dtype=np.float32), np.ones(len(FEATURE_COLS), dtype=np.float32)
    micro_mean = training_frame[MICRO_FEATURES].mean()
    micro_std = training_frame[MICRO_FEATURES].std(ddof=0)
    # Macro variables repeat for every stock. Fit their scaler on unique months,
    # so months with more listed firms do not receive more weight.
    macro_by_month = training_frame[['mthcaldt'] + MACRO_FEATURES].drop_duplicates('mthcaldt')
    macro_mean = macro_by_month[MACRO_FEATURES].mean()
    macro_std = macro_by_month[MACRO_FEATURES].std(ddof=0)
    means = pd.concat([micro_mean, macro_mean]).loc[FEATURE_COLS].to_numpy(dtype=np.float32)
    stds = pd.concat([micro_std, macro_std]).loc[FEATURE_COLS].to_numpy(dtype=np.float32)
    stds = np.where(stds < 1e-6, 1.0, stds)
    return means, stds

def transform_features(frame: pd.DataFrame, means: np.ndarray, stds: np.ndarray) -> np.ndarray:
    values = frame[FEATURE_COLS].to_numpy(dtype=np.float32, copy=True)
    return (values - means) / stds

def add_month_equal_weights(frame: pd.DataFrame) -> np.ndarray:
    counts = frame.groupby('mthcaldt')['permno'].transform('size').to_numpy(dtype=np.float32)
    weights = 1.0 / np.maximum(counts, 1.0)
    return weights / weights.mean()

def split_training_and_validation(history: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    dates = np.sort(history['mthcaldt'].unique())
    validation_count = min(VALIDATION_MONTHS, max(1, len(dates) // 4))
    validation_dates = set(dates[-validation_count:])
    train_frame = history.loc[~history['mthcaldt'].isin(validation_dates)].copy()
    validation_frame = history.loc[history['mthcaldt'].isin(validation_dates)].copy()
    if train_frame.empty or validation_frame.empty:
        raise ValueError('Not enough dates for an internal train/validation split.')
    return train_frame, validation_frame

def distribution_nll(model, x, y, weights):
    mu, sigma, nu = model(x)
    if model.distribution == 'normal':
        log_prob = torch.distributions.Normal(mu, sigma).log_prob(y)
    else:
        log_prob = torch.distributions.StudentT(nu, loc=mu, scale=sigma).log_prob(y)
    return -(weights * log_prob).sum() / weights.sum().clamp_min(1e-8)

def train_conditional_distribution(
    train_frame: pd.DataFrame, validation_frame: pd.DataFrame, distribution: str
):
    means, stds = fit_feature_scaler(train_frame)

    x_train = torch.tensor(transform_features(train_frame, means, stds), dtype=torch.float32)
    y_train = torch.tensor(train_frame['target_ret_final'].to_numpy(dtype=np.float32), dtype=torch.float32)
    w_train = torch.tensor(add_month_equal_weights(train_frame), dtype=torch.float32)
    train_loader = DataLoader(
        TensorDataset(x_train, y_train, w_train), batch_size=BATCH_SIZE, shuffle=True
    )

    x_validation = torch.tensor(transform_features(validation_frame, means, stds), dtype=torch.float32, device=device)
    y_validation = torch.tensor(validation_frame['target_ret_final'].to_numpy(dtype=np.float32), dtype=torch.float32, device=device)
    w_validation = torch.tensor(add_month_equal_weights(validation_frame), dtype=torch.float32, device=device)

    model = ConditionalDistributionNet(len(FEATURE_COLS), distribution=distribution).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )

    best_state = None
    best_validation_loss = np.inf
    patience_counter = 0

    for _ in range(MAX_EPOCHS):
        model.train()
        for x_batch, y_batch, w_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            w_batch = w_batch.to(device)
            optimizer.zero_grad()
            loss = distribution_nll(model, x_batch, y_batch, w_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            validation_loss = float(distribution_nll(
                model, x_validation, y_validation, w_validation
            ).cpu())
        if validation_loss < best_validation_loss - 1e-6:
            best_validation_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                break

    if best_state is None:
        raise RuntimeError('Training produced no finite validation loss. Check targets, features, and scales.')
    model.load_state_dict(best_state)
    return model, means, stds, best_validation_loss


: 

In [12]:
def iter_oos_blocks(panel: pd.DataFrame):
    oos_dates = np.array(sorted(
        panel.loc[panel['mthcaldt'].between(OOS_START, OOS_END), 'mthcaldt'].unique()
    ))
    for start_position in range(0, len(oos_dates), REFIT_MONTHS):
        block_dates = oos_dates[start_position:start_position + REFIT_MONTHS]
        if len(block_dates) == 0:
            continue
        block_start = pd.Timestamp(block_dates[0])
        history = panel.loc[panel['target_date'] < block_start].copy()
        test_frame = panel.loc[panel['mthcaldt'].isin(block_dates)].copy()
        yield start_position, block_start, history, test_frame

def run_conditional_parametric_oos(panel: pd.DataFrame, distribution: str) -> pd.DataFrame:
    model_id = 'conditional_normal_k1' if distribution == 'normal' else 'conditional_student_t_k1'
    output_rows = []

    for block_number, block_start, history, test_frame in iter_oos_blocks(panel):
        train_frame, validation_frame = split_training_and_validation(history)
        model, means, stds, validation_loss = train_conditional_distribution(
            train_frame, validation_frame, distribution
        )

        x_test = torch.tensor(
            transform_features(test_frame, means, stds), dtype=torch.float32, device=device
        )
        model.eval()
        with torch.no_grad():
            mu_tensor, sigma_tensor, nu_tensor = model(x_test)
        mu_values = mu_tensor.cpu().numpy()
        sigma_values = sigma_tensor.cpu().numpy()
        nu_values = None if nu_tensor is None else nu_tensor.cpu().numpy()

        for position, (_, row) in enumerate(test_frame.iterrows()):
            loc = float(mu_values[position])
            scale = float(sigma_values[position])
            output_row = base_forecast_row(row, model_id)
            if distribution == 'normal':
                pred_vol = scale
                tail_values = {
                    alpha: normal_var_and_loss_es(alpha, loc, scale) for alpha in ALPHAS
                }
                output_row.update({'loc': loc, 'scale': scale})
            else:
                nu = float(nu_values[position])
                pred_vol = scale * math.sqrt(nu / (nu - 2.0))
                tail_values = {
                    alpha: student_var_and_loss_es(alpha, loc, scale, nu) for alpha in ALPHAS
                }
                output_row.update({'loc': loc, 'scale': scale, 'nu': nu})
            output_row.update({
                'pred_mean': loc,
                'pred_vol': float(pred_vol),
                'validation_loss': float(validation_loss),
                'refit_date': block_start,
            })
            output_rows.append(add_tail_statistics(output_row, tail_values))

    return pd.DataFrame(output_rows) if output_rows else empty_forecast_frame()

if RUN_CONDITIONAL_NORMAL_K1:
    conditional_normal_forecasts = run_conditional_parametric_oos(panel_df, 'normal')
else:
    conditional_normal_forecasts = empty_forecast_frame()

if RUN_CONDITIONAL_STUDENT_K1:
    conditional_student_forecasts = run_conditional_parametric_oos(panel_df, 'student_t')
else:
    conditional_student_forecasts = empty_forecast_frame()

display(pd.concat([
    conditional_normal_forecasts, conditional_student_forecasts
]).groupby('model_id').size().rename('rows').to_frame())


Global seed set to 42 to ensure reproducibility.
Global seed set to 54 to ensure reproducibility.
Global seed set to 66 to ensure reproducibility.
Global seed set to 78 to ensure reproducibility.
Global seed set to 90 to ensure reproducibility.
Global seed set to 102 to ensure reproducibility.
Global seed set to 114 to ensure reproducibility.
Global seed set to 126 to ensure reproducibility.
Global seed set to 138 to ensure reproducibility.
Global seed set to 150 to ensure reproducibility.
Global seed set to 162 to ensure reproducibility.
Global seed set to 174 to ensure reproducibility.
Global seed set to 186 to ensure reproducibility.
Global seed set to 198 to ensure reproducibility.
Global seed set to 210 to ensure reproducibility.
Global seed set to 222 to ensure reproducibility.
Global seed set to 234 to ensure reproducibility.
Global seed set to 246 to ensure reproducibility.
Global seed set to 42 to ensure reproducibility.
Global seed set to 54 to ensure reproducibility.
Global 

,rows
model_id,
conditional_normal_k1,683379
conditional_student_t_k1,683379


: 

## 8. Quantile Benchmarks

The primary quantile benchmark is a joint **linear** quantile model trained with pinball loss. The optional MLP changes only the feature map; both models use the same quantile grid, time split, feature scaler, and walk-forward dates. Predicted quantiles are sorted at inference to remove finite-sample crossings.

For a quantile grid $Q(u)$, lower-tail ES is computed as

$$
LES_\alpha=-\frac{1}{\alpha}\int_0^\alpha Q(u)\,du.
$$


In [13]:
class JointQuantileNet(nn.Module):
    def __init__(self, input_dim: int, quantile_count: int, model_kind: str):
        super().__init__()
        if model_kind == 'linear':
            self.network = nn.Linear(input_dim, quantile_count)
        elif model_kind == 'mlp':
            self.network = nn.Sequential(
                nn.Linear(input_dim, HIDDEN_DIMS[0]),
                nn.LayerNorm(HIDDEN_DIMS[0]),
                nn.ELU(),
                nn.Dropout(DROPOUT),
                nn.Linear(HIDDEN_DIMS[0], HIDDEN_DIMS[1]),
                nn.LayerNorm(HIDDEN_DIMS[1]),
                nn.ELU(),
                nn.Linear(HIDDEN_DIMS[1], quantile_count),
            )
        else:
            raise ValueError('model_kind must be linear or mlp')

    def forward(self, x):
        return self.network(x)

def weighted_pinball_loss(predicted_quantiles, targets, weights, quantile_tensor):
    errors = targets.unsqueeze(1) - predicted_quantiles
    pinball = torch.maximum(quantile_tensor * errors, (quantile_tensor - 1.0) * errors)
    row_loss = pinball.mean(dim=1)
    return (weights * row_loss).sum() / weights.sum().clamp_min(1e-8)

def train_quantile_model(
    train_frame: pd.DataFrame, validation_frame: pd.DataFrame, model_kind: str
):
    # seed_everything(block_seed)
    means, stds = fit_feature_scaler(train_frame)
    quantile_tensor = torch.tensor(QUANTILE_LEVELS, dtype=torch.float32, device=device).view(1, -1)

    x_train = torch.tensor(transform_features(train_frame, means, stds), dtype=torch.float32)
    y_train = torch.tensor(train_frame['target_ret_final'].to_numpy(dtype=np.float32), dtype=torch.float32)
    w_train = torch.tensor(add_month_equal_weights(train_frame), dtype=torch.float32)
    train_loader = DataLoader(
        TensorDataset(x_train, y_train, w_train), batch_size=BATCH_SIZE, shuffle=True
    )

    x_validation = torch.tensor(transform_features(validation_frame, means, stds), dtype=torch.float32, device=device)
    y_validation = torch.tensor(validation_frame['target_ret_final'].to_numpy(dtype=np.float32), dtype=torch.float32, device=device)
    w_validation = torch.tensor(add_month_equal_weights(validation_frame), dtype=torch.float32, device=device)

    model = JointQuantileNet(len(FEATURE_COLS), len(QUANTILE_LEVELS), model_kind).to(device)
    output_layer = model.network if model_kind == 'linear' else model.network[-1]
    initial_quantiles = np.quantile(train_frame['target_ret_final'], QUANTILE_LEVELS)
    nn.init.zeros_(output_layer.weight)
    with torch.no_grad():
        output_layer.bias.copy_(torch.tensor(initial_quantiles, dtype=torch.float32, device=device))
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )

    best_state = None
    best_validation_loss = np.inf
    patience_counter = 0

    for _ in range(MAX_EPOCHS):
        model.train()
        for x_batch, y_batch, w_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            w_batch = w_batch.to(device)
            optimizer.zero_grad()
            predicted = model(x_batch)
            loss = weighted_pinball_loss(predicted, y_batch, w_batch, quantile_tensor)
            crossing_penalty = F.relu(predicted[:, :-1] - predicted[:, 1:]).mean()
            total_loss = loss + 0.10 * crossing_penalty
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            validation_prediction = model(x_validation)
            validation_loss = float(weighted_pinball_loss(
                validation_prediction, y_validation, w_validation, quantile_tensor
            ).cpu())
        if validation_loss < best_validation_loss - 1e-6:
            best_validation_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                break

    if best_state is None:
        raise RuntimeError('Quantile training produced no finite validation loss.')
    model.load_state_dict(best_state)
    return model, means, stds, best_validation_loss

def trapezoid_integral(values: np.ndarray, levels: np.ndarray) -> float:
    # NumPy >=2.0 prefers trapezoid; older Colab images expose trapz only.
    integrate = getattr(np, 'trapezoid', None)
    if integrate is None:
        integrate = np.trapz
    return float(integrate(values, levels))

def quantile_grid_moments(quantile_values: np.ndarray) -> tuple[float, float]:
    quantile_values = np.sort(np.asarray(quantile_values, dtype=float))
    full_levels = np.concatenate(([0.0], QUANTILE_LEVELS, [1.0]))
    full_quantiles = np.concatenate(([quantile_values[0]], quantile_values, [quantile_values[-1]]))
    mean_value = trapezoid_integral(full_quantiles, full_levels)
    second_moment = trapezoid_integral(full_quantiles ** 2, full_levels)
    variance = max(second_moment - mean_value ** 2, 0.0)
    return mean_value, math.sqrt(variance)

def quantile_grid_var_and_loss_es(
    quantile_values: np.ndarray, alpha: float
) -> tuple[float, float]:
    quantile_values = np.sort(np.asarray(quantile_values, dtype=float))
    var_value = float(np.interp(alpha, QUANTILE_LEVELS, quantile_values))
    interior_mask = QUANTILE_LEVELS < alpha
    integration_levels = np.concatenate(([0.0], QUANTILE_LEVELS[interior_mask], [alpha]))
    integration_quantiles = np.concatenate((
        [quantile_values[0]], quantile_values[interior_mask], [var_value]
    ))
    tail_mean = trapezoid_integral(integration_quantiles, integration_levels) / alpha
    return var_value, -tail_mean


: 

In [14]:
def run_quantile_oos(panel: pd.DataFrame, model_kind: str) -> pd.DataFrame:
    model_id = 'linear_quantile' if model_kind == 'linear' else 'quantile_mlp'
    output_rows = []

    for block_number, block_start, history, test_frame in iter_oos_blocks(panel):
        train_frame, validation_frame = split_training_and_validation(history)
        model, means, stds, validation_loss = train_quantile_model(
            train_frame, validation_frame, model_kind
        )
        x_test = torch.tensor(
            transform_features(test_frame, means, stds), dtype=torch.float32, device=device
        )
        model.eval()
        with torch.no_grad():
            predicted_quantiles = model(x_test).cpu().numpy()
        predicted_quantiles = np.sort(predicted_quantiles, axis=1)

        for position, (_, row) in enumerate(test_frame.iterrows()):
            quantile_values = predicted_quantiles[position]
            pred_mean, pred_vol = quantile_grid_moments(quantile_values)
            tail_values = {
                alpha: quantile_grid_var_and_loss_es(quantile_values, alpha)
                for alpha in ALPHAS
            }
            output_row = base_forecast_row(row, model_id)
            output_row.update({
                'pred_mean': pred_mean,
                'pred_vol': pred_vol,
                'quantile_levels': json.dumps(QUANTILE_LEVELS.tolist()),
                'quantile_vec': json.dumps(quantile_values.tolist()),
                'validation_loss': float(validation_loss),
                'refit_date': block_start,
            })
            output_rows.append(add_tail_statistics(output_row, tail_values))

    return pd.DataFrame(output_rows) if output_rows else empty_forecast_frame()

if RUN_LINEAR_QUANTILE:
    linear_quantile_forecasts = run_quantile_oos(panel_df, 'linear')
else:
    linear_quantile_forecasts = empty_forecast_frame()

if RUN_QUANTILE_MLP:
    quantile_mlp_forecasts = run_quantile_oos(panel_df, 'mlp')
else:
    quantile_mlp_forecasts = empty_forecast_frame()

display(pd.concat([
    linear_quantile_forecasts, quantile_mlp_forecasts
]).groupby('model_id').size().rename('rows').to_frame())


,rows
model_id,
linear_quantile,683379


: 

## 9. Combine, Validate, and Save

This section performs structural checks only. NLL, CRPS, PIT, VaR coverage, and joint VaR--ES model selection belong in `5PIT_CRPS_Test.ipynb`.


In [15]:
forecast_frames = [
    rolling_forecasts,
    conditional_normal_forecasts,
    conditional_student_forecasts,
    linear_quantile_forecasts,
    quantile_mlp_forecasts,
]
baseline_forecasts = pd.concat(
    [frame for frame in forecast_frames if not frame.empty], ignore_index=True, sort=False
)
baseline_forecasts['forecast_date'] = pd.to_datetime(baseline_forecasts['forecast_date'])
baseline_forecasts['target_date'] = pd.to_datetime(baseline_forecasts['target_date'])
baseline_forecasts = baseline_forecasts.sort_values(
    ['model_id', 'forecast_date', 'permno']
).reset_index(drop=True)

required_output_cols = [
    'permno', 'forecast_date', 'target_date', 'realized_ret', 'model_id',
    'pred_mean', 'pred_vol', 'var_1', 'es_1',
    'var_5', 'es_5', 'var_10', 'es_10'
]
missing_output_cols = [col for col in required_output_cols if col not in baseline_forecasts.columns]
if missing_output_cols:
    raise KeyError(f'Unified output is missing columns: {missing_output_cols}')

if baseline_forecasts.duplicated(['model_id', 'permno', 'forecast_date']).any():
    raise ValueError('Duplicate model-stock-forecast_date rows found.')

finite_cols = ['pred_mean', 'pred_vol', 'var_1', 'es_1', 'var_5', 'es_5', 'var_10', 'es_10']
if not np.isfinite(baseline_forecasts[finite_cols].to_numpy(dtype=float)).all():
    raise ValueError('Non-finite distribution statistics found in baseline output.')

tolerance = 1e-8
var_order_ok = (
    (baseline_forecasts['var_1'] <= baseline_forecasts['var_5'] + tolerance)
    & (baseline_forecasts['var_5'] <= baseline_forecasts['var_10'] + tolerance)
).mean()
es_order_ok = (
    (baseline_forecasts['es_1'] + tolerance >= baseline_forecasts['es_5'])
    & (baseline_forecasts['es_5'] + tolerance >= baseline_forecasts['es_10'])
).mean()

summary = baseline_forecasts.groupby('model_id').agg(
    rows=('permno', 'size'),
    first_forecast=('forecast_date', 'min'),
    last_forecast=('forecast_date', 'max'),
    mean_predicted_return=('pred_mean', 'mean'),
    mean_predicted_volatility=('pred_vol', 'mean'),
    mean_es_5=('es_5', 'mean'),
).sort_index()

print(f'VaR ordering pass rate: {var_order_ok:.4%}')
print(f'Loss-ES ordering pass rate: {es_order_ok:.4%}')
display(summary)
display(baseline_forecasts[required_output_cols].head(10))

if var_order_ok < 0.999 or es_order_ok < 0.999:
    raise AssertionError('VaR/ES alpha ordering check failed for too many rows.')



VaR ordering pass rate: 100.0000%
Loss-ES ordering pass rate: 100.0000%


,rows,first_forecast,last_forecast,mean_predicted_return,mean_predicted_volatility,mean_es_5
model_id,,,,,,
conditional_normal_k1,683379,2007-01-31,2024-12-31,0.007449,0.119318,0.238670
conditional_student_t_k1,683379,2007-01-31,2024-12-31,0.004316,0.116806,0.255291
historical_empirical,560591,2007-01-31,2024-12-31,0.017128,0.122242,0.221543
linear_quantile,683379,2007-01-31,2024-12-31,0.006681,0.129071,0.252873
rolling_normal,560591,2007-01-31,2024-12-31,0.017128,0.122242,0.235022
rolling_student_t_mom,560591,2007-01-31,2024-12-31,0.017128,0.122242,0.250243


,permno,forecast_date,target_date,realized_ret,model_id,pred_mean,pred_vol,var_1,es_1,var_5,es_5,var_10,es_10
0,10001,2007-01-31,2007-02-28,0.264774,conditional_normal_k1,0.007201,0.110076,-0.248873,0.286174,-0.173857,0.219853,-0.133866,0.185979
1,10002,2007-01-31,2007-02-28,0.033054,conditional_normal_k1,0.002971,0.081335,-0.186243,0.213805,-0.130813,0.164800,-0.101264,0.139771
2,10025,2007-01-31,2007-02-28,-0.028094,conditional_normal_k1,0.010869,0.142247,-0.320048,0.368250,-0.223107,0.282546,-0.171428,0.238773
3,10026,2007-01-31,2007-02-28,-0.038517,conditional_normal_k1,0.017396,0.084474,-0.179119,0.207744,-0.121551,0.156849,-0.090861,0.130854
4,10032,2007-01-31,2007-02-28,-0.023810,conditional_normal_k1,-0.007038,0.162429,-0.384906,0.439948,-0.274211,0.342084,-0.215200,0.292099
5,10044,2007-01-31,2007-02-28,0.021561,conditional_normal_k1,0.006813,0.115671,-0.262279,0.301476,-0.183449,0.231784,-0.141426,0.196188
6,10051,2007-01-31,2007-02-28,0.303911,conditional_normal_k1,0.007572,0.107149,-0.241693,0.278002,-0.168672,0.213445,-0.129744,0.180472
7,10078,2007-01-31,2007-02-28,-0.076807,conditional_normal_k1,0.005055,0.090126,-0.204610,0.235151,-0.143190,0.180850,-0.110447,0.153115
8,10092,2007-01-31,2007-02-28,-0.050077,conditional_normal_k1,0.015274,0.098916,-0.214839,0.248358,-0.147428,0.188761,-0.111492,0.158322
9,10104,2007-01-31,2007-02-28,-0.042541,conditional_normal_k1,0.009442,0.069976,-0.153347,0.177060,-0.105659,0.134899,-0.080236,0.113365


: 

In [16]:
baseline_forecasts.sample(10)

,permno,forecast_date,target_date,realized_ret,model_id,pred_mean,pred_vol,history_n,var_1,es_1,var_5,es_5,var_10,es_10,loc,scale,nu,validation_loss,refit_date,quantile_levels,quantile_vec
3605717,68292,2020-10-31,2020-11-30,0.192529,rolling_student_t_mom,0.009765,0.074922,192.0,-0.181444,0.233383,-0.109612,0.155407,-0.078995,0.124019,0.009765,0.062122,6.400302,NaN,NaT,NaN,NaN
44761,76374,2007-12-31,2008-01-31,-0.083453,conditional_normal_k1,0.029137,0.207631,NaN,-0.453885,0.524244,-0.312386,0.399146,-0.236953,0.335252,0.029137,0.207631,NaN,-0.946316,2007-01-31,NaN,NaN
1863956,16698,2022-11-30,2022-12-31,-0.143834,historical_empirical,0.034667,0.230519,67.0,-0.252514,0.253342,-0.222589,0.241838,-0.206018,0.229785,NaN,NaN,NaN,NaN,NaT,NaN,NaN
3580167,86996,2019-11-30,2019-12-31,0.019115,rolling_student_t_mom,0.025163,0.084244,185.0,-0.189179,0.246085,-0.109359,0.160145,-0.075048,0.125244,0.025163,0.070432,6.644079,NaN,NaT,NaN,NaN
3280250,11382,2010-04-30,2010-05-31,-0.058462,rolling_student_t_mom,0.023433,0.185159,185.0,-0.466180,0.652498,-0.257626,0.395358,-0.179342,0.304544,0.023433,0.133119,4.139767,NaN,NaT,NaN,NaN
3216117,60871,2008-03-31,2008-04-30,0.091125,rolling_student_t_mom,0.025764,0.147241,192.0,-0.354247,0.466952,-0.206635,0.301505,-0.145744,0.237136,0.025764,0.118102,5.607897,NaN,NaT,NaN,NaN
1421820,88159,2008-06-30,2008-07-31,0.400000,historical_empirical,0.007646,0.183255,96.0,-0.318331,0.342776,-0.285769,0.313854,-0.207612,0.277962,NaN,NaN,NaN,NaN,NaT,NaN,NaN
1989361,11599,2008-05-31,2008-06-30,-0.077771,linear_quantile,0.039496,0.114382,NaN,-0.154334,0.256603,-0.095777,0.147694,-0.079777,0.117735,NaN,NaN,NaN,0.011503,2008-01-31,"[0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1,...","[-0.4303893446922302, -0.270126074552536, -0.2..."
2343780,76841,2017-12-31,2018-01-31,0.091785,linear_quantile,0.012531,0.070570,NaN,-0.142994,0.198834,-0.099089,0.133152,-0.071893,0.109321,NaN,NaN,NaN,0.012125,2017-01-31,"[0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1,...","[-0.2837212383747101, -0.2226838320493698, -0...."
386992,92322,2017-02-28,2017-03-31,0.043156,conditional_normal_k1,-0.004578,0.069877,NaN,-0.167137,0.190816,-0.119516,0.148715,-0.094129,0.127211,-0.004578,0.069877,NaN,-0.837211,2017-01-31,NaN,NaN


## 10. Handoff to Forecast Evaluation

Before the real run:

1. choose `RUN_MODE='quick'` for a bounded test or `RUN_MODE='full'` for production;
2. on Kaggle, attach the dataset containing both parquet files; on Colab, place them under `COLAB_DATA_DIR`;
3. confirm the explicit micro/macro feature lists and final OOS dates;
4. run top-to-bottom and keep the generated config JSON with the Parquet output.

The next notebook should compare these forecasts with MDN forecasts using monthly aggregated NLL/CRPS/tail-CRPS, PIT diagnostics, VaR coverage, and a joint VaR--ES score. Do not select a benchmark or MDN using portfolio returns.


: 